In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ball_by_ball = pd.read_csv('/kaggle/input/ipl-ball-by-ball-dataset-2008-2025/ball_by_ball_ipl_data.csv')
match_info = pd.read_csv('/kaggle/input/ipl-ball-by-ball-dataset-2008-2025/basic_info.csv')

print(f"Ball-by-ball: {ball_by_ball.shape[0]:,} rows × {ball_by_ball.shape[1]} columns")
print(f"Match info: {match_info.shape[0]:,} rows × {match_info.shape[1]} columns")



Ball-by-ball: 278,205 rows × 10 columns
Match info: 1,169 rows × 9 columns


In [3]:
print("="*50)
print("DATA QUALITY CHECKS")
print("="*50)

print("\nMissing Values - Ball by Ball:")
print(ball_by_ball.isnull().sum())
print("\nMissing Values - Match Info:")
print(match_info.isnull().sum())

print("\nDuplicate Rows:")
print(f"Ball-by-ball duplicates: {ball_by_ball.duplicated().sum()}")
print(f"Match info duplicates: {match_info.duplicated().sum()}")



DATA QUALITY CHECKS

Missing Values - Ball by Ball:
Unnamed: 0            0
id                    0
batter_name           0
bowler_name           0
non_striker_name      0
batsman_run           0
extra_run             0
total_run             0
batting_team          0
bowling_team        495
dtype: int64

Missing Values - Match Info:
Unnamed: 0        0
id                0
city             51
gender            0
pom               8
toss_decision     0
winner            0
team_type         0
won_by           23
dtype: int64

Duplicate Rows:
Ball-by-ball duplicates: 0
Match info duplicates: 0


In [4]:
print("\nData Types - Ball by Ball:")
print(ball_by_ball.dtypes)
print("\nData Types - Match Info:")
print(match_info.dtypes)

print("\nBasic Statistics - Ball by Ball:")
print(ball_by_ball.describe())
print("\nBasic Statistics - Match Info:")
print(match_info.describe())




Data Types - Ball by Ball:
Unnamed: 0           int64
id                   int64
batter_name         object
bowler_name         object
non_striker_name    object
batsman_run          int64
extra_run            int64
total_run            int64
batting_team        object
bowling_team        object
dtype: object

Data Types - Match Info:
Unnamed: 0        int64
id                int64
city             object
gender           object
pom              object
toss_decision    object
winner           object
team_type        object
won_by           object
dtype: object

Basic Statistics - Ball by Ball:
          Unnamed: 0             id    batsman_run      extra_run  \
count  278205.000000  278205.000000  278205.000000  278205.000000   
mean   139102.000000     582.222778       1.277378       0.067971   
std     80311.010157     336.911762       1.651107       0.343033   
min         0.000000       0.000000       0.000000       0.000000   
25%     69551.000000     291.000000       0.000000   

In [5]:
print("="*50)
print("HANDLING MISSING VALUES")
print("="*50)

missing_ball = ball_by_ball.isnull().sum()
missing_match = match_info.isnull().sum()

if missing_ball.sum() > 0:
    print("\nMissing values found in ball_by_ball:")
    print(missing_ball[missing_ball > 0])
    ball_by_ball_clean = ball_by_ball.dropna()
    print(f"\nRows after dropping missing values: {ball_by_ball_clean.shape[0]}")
else:
    print("\nNo missing values in ball_by_ball")
    ball_by_ball_clean = ball_by_ball.copy()

if missing_match.sum() > 0:
    print("\nMissing values found in match_info:")
    print(missing_match[missing_match > 0])
    match_info_clean = match_info.dropna()
    print(f"\nRows after dropping missing values: {match_info_clean.shape[0]}")
else:
    print("\nNo missing values in match_info")
    match_info_clean = match_info.copy()



HANDLING MISSING VALUES

Missing values found in ball_by_ball:
bowling_team    495
dtype: int64

Rows after dropping missing values: 277710

Missing values found in match_info:
city      51
pom        8
won_by    23
dtype: int64

Rows after dropping missing values: 1098


In [6]:
print("="*50)
print("HANDLING DUPLICATES")
print("="*50)

if ball_by_ball_clean.duplicated().sum() > 0:
    print(f"\nRemoving {ball_by_ball_clean.duplicated().sum()} duplicates from ball_by_ball")
    ball_by_ball_clean = ball_by_ball_clean.drop_duplicates()
else:
    print("\nNo duplicates in ball_by_ball")

if match_info_clean.duplicated().sum() > 0:
    print(f"\nRemoving {match_info_clean.duplicated().sum()} duplicates from match_info")
    match_info_clean = match_info_clean.drop_duplicates()
else:
    print("\nNo duplicates in match_info")

print(f"\nFinal ball_by_ball shape: {ball_by_ball_clean.shape}")
print(f"Final match_info shape: {match_info_clean.shape}")



HANDLING DUPLICATES

No duplicates in ball_by_ball

No duplicates in match_info

Final ball_by_ball shape: (277710, 10)
Final match_info shape: (1098, 9)


In [7]:
print("="*50)
print("DATA TYPE VALIDATION")
print("="*50)

print("\nChecking numerical columns:")
numeric_cols = ['batsman_run', 'extra_run', 'total_run']
for col in numeric_cols:
    if col in ball_by_ball_clean.columns:
        print(f"{col}: {ball_by_ball_clean[col].dtype}")
        if ball_by_ball_clean[col].dtype == 'object':
            ball_by_ball_clean[col] = pd.to_numeric(ball_by_ball_clean[col], errors='coerce')
            print(f"  Converted to: {ball_by_ball_clean[col].dtype}")



DATA TYPE VALIDATION

Checking numerical columns:
batsman_run: int64
extra_run: int64
total_run: int64


In [8]:
print("="*50)
print("OUTLIER DETECTION")
print("="*50)

print("\nOutlier Analysis using IQR method:")
for col in ['batsman_run', 'extra_run', 'total_run']:
    Q1 = ball_by_ball_clean[col].quantile(0.25)
    Q3 = ball_by_ball_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = ball_by_ball_clean[(ball_by_ball_clean[col] < lower_bound) | (ball_by_ball_clean[col] > upper_bound)]
    print(f"\n{col}:")
    print(f"  Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
    print(f"  Lower bound: {lower_bound}, Upper bound: {upper_bound}")
    print(f"  Outliers: {len(outliers)} ({len(outliers)/len(ball_by_ball_clean)*100:.2f}%)")
    print(f"  Min: {ball_by_ball_clean[col].min()}, Max: {ball_by_ball_clean[col].max()}")



OUTLIER DETECTION

Outlier Analysis using IQR method:

batsman_run:
  Q1: 0.0, Q3: 1.0, IQR: 1.0
  Lower bound: -1.5, Upper bound: 2.5
  Outliers: 47264 (17.02%)
  Min: 0, Max: 6

extra_run:
  Q1: 0.0, Q3: 0.0, IQR: 0.0
  Lower bound: 0.0, Upper bound: 0.0
  Outliers: 15106 (5.44%)
  Min: 0, Max: 7

total_run:
  Q1: 0.0, Q3: 1.0, IQR: 1.0
  Lower bound: -1.5, Upper bound: 2.5
  Outliers: 48285 (17.39%)
  Min: 0, Max: 7


In [10]:
print("="*50)
print("SAVING CLEANED DATA")
print("="*50)

ball_by_ball_clean.to_csv('ball_by_ball_cleaned.csv', index=False)
match_info_clean.to_csv('match_info_cleaned.csv', index=False)

print("\nCleaned datasets saved:")
print("- ball_by_ball_cleaned.csv")
print("- match_info_cleaned.csv")


SAVING CLEANED DATA

Cleaned datasets saved:
- ball_by_ball_cleaned.csv
- match_info_cleaned.csv
